###  Setup e Configuração
Verifica a existência da variáveis de ambiente, valida a conexão com ADLS e lista os arquivos dispiníveis no container raw/batch-data.

In [0]:
# Instalação de dependências
%pip install python-dotenv azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
# Verifica a existência de um arquivo .env
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis de Ambiente encontradas no caminho:{dotenv_path} ")
else:
    print("Variáveis de Ambiente não encontradas.")

In [0]:
# Cria um dicionário com as credenciais
adls_credentials = {
    "CLIENT_ID": os.getenv("CLIENT_ID"),
    "TENANT_ID": os.getenv("TENANT_ID"),
    "CLIENT_SECRET": os.getenv("CLIENT_SECRET"),
    "STORAGE_ACCOUNT_NAME": os.getenv("STORAGE_ACCOUNT_NAME"),
    "CONTAINER_NAME": os.getenv("CONTAINER_NAME", "raw")
}

jdbc_credentials = {
    "JDBC_HOST": os.getenv("JDBC_HOST"),
    "JDBC_DATABASE": os.getenv("JDBC_DATABASE"),
    "JDBC_USERNAME": os.getenv("JDBC_USERNAME"),
    "JDBC_PASSWORD": os.getenv("JDBC_PASSWORD")
}

# Percorre cada credencial e verifica se está definida
print("\nStatus das Credenciais ADLS")
for k, v in adls_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

print("\nStatus das Credenciais SQL Server")
for k, v in jdbc_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")


In [0]:
# TESTE DE CONEXÃO COM ADLS — MÉTODO VIA SPARK / ABFS

# Dicionário de Credenciais OAuth para usar com o Spark
adls_options = {
    f"fs.azure.account.auth.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_credentials["TENANT_ID"]}/oauth2/token"
}

# Define o Caminho para a pasta do Container
batch_data_path = (f"abfss://{adls_credentials["CONTAINER_NAME"]}@{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net/""batch-data")

# Lista os arquivos do path
files = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(batch_data_path)
    .select("path")
)
files.show(truncate=False)

In [0]:
# TESTE DE CONEXÃO COM ADLS — MÉTODO VIA AZURE SDK 
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from azure.storage.blob import BlobServiceClient


# Autenticação e inicialização dos clientes DataLakeServiceClient e BlobServiceClient
credential = ClientSecretCredential(adls_credentials["TENANT_ID"], adls_credentials["CLIENT_ID"], adls_credentials["CLIENT_SECRET"])
service_client = DataLakeServiceClient(account_url=f"https://{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net",credential=credential,)
blob_service_client = BlobServiceClient(account_url=f"https://{adls_credentials["STORAGE_ACCOUNT_NAME"]}.blob.core.windows.net",credential=credential)

# Lista os containers disponíveis na Storage Account
containers_encontrados = []
for container in blob_service_client.list_containers():
    print(container.name)
    containers_encontrados.append(container.name)
print(f"\nTotal de containers encontrados: {len(containers_encontrados)}")

# Lista o conteúdo de cada container
for container_name in containers_encontrados:
    fs_client = service_client.get_file_system_client(container_name)

    arquivos_encontrados = []
    print(f"\nConteúdo do Container {container_name}")
    for p in fs_client.get_paths():
        print(p.name)
        arquivos_encontrados.append(p.name)
    print(f"\nTotal de itens encontrados: {len(arquivos_encontrados)}")
